In [1]:
# # Notebook 01 — From EDA-Processed to Model-Ready
# **LCGA Self-Healing IDS | Addis Ababa University**
# 
# Loads the .npy files saved by the EDA notebook, then:
# - Splits into train/val/test (before any further processing)
# - Feature selection on train only
# - Scaling on train only  
# - SMOTE on train only
# - Sliding window sequences for GRU/LSTM

In [2]:
import os

base = "/kaggle/input/notebooks/getayefiseha/eda-for-nlskdd-cic-ids-ipynb"
for root, dirs, files in os.walk(base):
    for f in files:
        if f.endswith(".npy") or f.endswith(".pkl"):
            print(os.path.join(root, f))

/kaggle/input/notebooks/getayefiseha/eda-for-nlskdd-cic-ids-ipynb/processed/cic_classes.pkl
/kaggle/input/notebooks/getayefiseha/eda-for-nlskdd-cic-ids-ipynb/processed/cic_scaler.pkl
/kaggle/input/notebooks/getayefiseha/eda-for-nlskdd-cic-ids-ipynb/processed/cic_feature_names.pkl
/kaggle/input/notebooks/getayefiseha/eda-for-nlskdd-cic-ids-ipynb/processed/nsl_X.npy
/kaggle/input/notebooks/getayefiseha/eda-for-nlskdd-cic-ids-ipynb/processed/cic_label_enc.pkl
/kaggle/input/notebooks/getayefiseha/eda-for-nlskdd-cic-ids-ipynb/processed/nsl_scaler.pkl
/kaggle/input/notebooks/getayefiseha/eda-for-nlskdd-cic-ids-ipynb/processed/nsl_feature_names.pkl
/kaggle/input/notebooks/getayefiseha/eda-for-nlskdd-cic-ids-ipynb/processed/nsl_y.npy
/kaggle/input/notebooks/getayefiseha/eda-for-nlskdd-cic-ids-ipynb/processed/cic_y.npy
/kaggle/input/notebooks/getayefiseha/eda-for-nlskdd-cic-ids-ipynb/processed/cic_X.npy
/kaggle/input/notebooks/getayefiseha/eda-for-nlskdd-cic-ids-ipynb/processed/nsl_label_enc.pk

In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, RobustScaler, LabelEncoder
from sklearn.feature_selection import RFE
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
import os, joblib, warnings
warnings.filterwarnings("ignore")

# ── Paths ──
IN_DIR  = "/kaggle/input/notebooks/getayefiseha/eda-for-nlskdd-cic-ids-ipynb/processed"
OUT_DIR = "/kaggle/working/processed"
os.makedirs(OUT_DIR, exist_ok=True)
print("Imports OK")

Imports OK


# ## A. NSL-KDD Preprocessing

In [4]:
# ==================== A.1 Load ====================
X_nsl_raw = np.load(os.path.join(IN_DIR, "nsl_X.npy"))       # (147907, 40)
y_nsl_raw = np.load(os.path.join(IN_DIR, "nsl_y.npy"))       # (147907,) binary
feature_names_nsl = joblib.load(os.path.join(IN_DIR, "nsl_feature_names.pkl"))

print(f"NSL‑KDD: X={X_nsl_raw.shape}, y={y_nsl_raw.shape}")
print(f"Labels: {np.unique(y_nsl_raw, return_counts=True)}")

NSL‑KDD: X=(147907, 40), y=(147907,)
Labels: (array([0, 1]), array([70940, 76967]))


In [5]:
# ==================== A.2 Train/Val/Test Split (Stratified) ====================
X_train_nsl, X_temp, y_train_nsl, y_temp = train_test_split(
    X_nsl_raw, y_nsl_raw, test_size=0.30, stratify=y_nsl_raw, random_state=42
)
X_val_nsl, X_test_nsl, y_val_nsl, y_test_nsl = train_test_split(
    X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=42
)

print(f"Train: {X_train_nsl.shape} | Val: {X_val_nsl.shape} | Test: {X_test_nsl.shape}")
print(f"Train class dist: {np.bincount(y_train_nsl)}")

Train: (103534, 40) | Val: (22186, 40) | Test: (22187, 40)
Train class dist: [49658 53876]


In [6]:
# ==================== A.3 Feature Selection (RFE top 20) on Train Only ====================
print("Running RFE (≈3 min) …")
rf_rfe = RandomForestClassifier(n_estimators=50, random_state=42, n_jobs=-1)
rfe = RFE(rf_rfe, n_features_to_select=20, step=1)
rfe.fit(X_train_nsl, y_train_nsl)
rfe_mask = rfe.support_

X_train_sel = X_train_nsl[:, rfe_mask]
X_val_sel   = X_val_nsl[:, rfe_mask]
X_test_sel  = X_test_nsl[:, rfe_mask]

selected_names_nsl = [feature_names_nsl[i] for i, s in enumerate(rfe_mask) if s]
print(f"Selected {len(selected_names_nsl)} features: {selected_names_nsl}")

Running RFE (≈3 min) …
Selected 20 features: ['duration', 'src_bytes', 'dst_bytes', 'hot', 'logged_in', 'count', 'srv_count', 'rerror_rate', 'same_srv_rate', 'diff_srv_rate', 'dst_host_count', 'dst_host_srv_count', 'dst_host_same_srv_rate', 'dst_host_diff_srv_rate', 'dst_host_same_src_port_rate', 'dst_host_srv_diff_host_rate', 'dst_host_serror_rate', 'dst_host_rerror_rate', 'dst_host_srv_rerror_rate', 'total_bytes']


In [7]:
# ==================== A.4 Re‑Scale after Feature Selection (fit on Train only) ====================
scaler_nsl = StandardScaler()
X_train_sc = scaler_nsl.fit_transform(X_train_sel)
X_val_sc   = scaler_nsl.transform(X_val_sel)
X_test_sc  = scaler_nsl.transform(X_test_sel)
print("Re‑scaled after feature selection")

Re‑scaled after feature selection


In [8]:
# ==================== A.5 SMOTE on Train Only ====================
smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train_sc, y_train_nsl)
print(f"After SMOTE: {X_train_res.shape} | Class dist: {np.bincount(y_train_res)}")

After SMOTE: (107752, 20) | Class dist: [53876 53876]


In [9]:
# ==================== A.6 Sliding Window Sequences (window=10) for GRU ====================
def make_sequences(X, window=10):
    n, f = X.shape
    pad = np.zeros((window-1, f), dtype=X.dtype)
    Xp  = np.vstack([pad, X])
    seqs = np.lib.stride_tricks.sliding_window_view(Xp, window_shape=window, axis=0)
    return seqs.transpose(0, 2, 1)   # (n, window, f)

X_train_seq = make_sequences(X_train_res, 10)
X_val_seq   = make_sequences(X_val_sc,   10)
X_test_seq  = make_sequences(X_test_sc,  10)
print(f"Train seq: {X_train_seq.shape} | Val seq: {X_val_seq.shape} | Test seq: {X_test_seq.shape}")

Train seq: (107752, 10, 20) | Val seq: (22186, 10, 20) | Test seq: (22187, 10, 20)


In [10]:
# ==================== A.7 Save NSL‑KDD Model‑Ready Artifacts ====================
np.save(os.path.join(OUT_DIR, "nsl_Xtrain_seq.npy"),  X_train_seq)
np.save(os.path.join(OUT_DIR, "nsl_Xtrain_flat.npy"), X_train_res)
np.save(os.path.join(OUT_DIR, "nsl_ytrain.npy"),       y_train_res)
np.save(os.path.join(OUT_DIR, "nsl_Xval_seq.npy"),     X_val_seq)
np.save(os.path.join(OUT_DIR, "nsl_Xval_flat.npy"),    X_val_sc)
np.save(os.path.join(OUT_DIR, "nsl_yval.npy"),          y_val_nsl)
np.save(os.path.join(OUT_DIR, "nsl_Xtest_seq.npy"),    X_test_seq)
np.save(os.path.join(OUT_DIR, "nsl_Xtest_flat.npy"),   X_test_sc)
np.save(os.path.join(OUT_DIR, "nsl_ytest.npy"),         y_test_nsl)

joblib.dump(scaler_nsl,         os.path.join(OUT_DIR, "nsl_scaler.pkl"))
joblib.dump(rfe_mask,           os.path.join(OUT_DIR, "nsl_rfe_mask.pkl"))
joblib.dump(selected_names_nsl, os.path.join(OUT_DIR, "nsl_selected_features.pkl"))
print("NSL‑KDD model‑ready artifacts saved ✓")

NSL‑KDD model‑ready artifacts saved ✓


# ## B. CICIDS2017 Preprocessing

In [11]:
# ==================== B.1 Load ====================
X_cic_raw = np.load(os.path.join(IN_DIR, "cic_X.npy"))          # (2518655, 73)
y_cic_raw = np.load(os.path.join(IN_DIR, "cic_y.npy"))          # (2518655,)
feature_names_cic = joblib.load(os.path.join(IN_DIR, "cic_feature_names.pkl"))
class_names_cic   = joblib.load(os.path.join(IN_DIR, "cic_classes.pkl"))

print(f"CICIDS2017: X={X_cic_raw.shape}, y={y_cic_raw.shape}")
print(f"Features: {len(feature_names_cic)} | Classes: {len(class_names_cic)}")
print(f"Classes: {class_names_cic}")

CICIDS2017: X=(2518655, 73), y=(2518655,)
Features: 73 | Classes: 12
Classes: ['BENIGN', 'Bot', 'DDoS', 'DoS GoldenEye', 'DoS Hulk', 'DoS Slowhttptest', 'DoS slowloris', 'FTP-Patator', 'Heartbleed', 'Infiltration', 'PortScan', 'SSH-Patator']


In [12]:
# ==================== B.2 Stratified Train/Val/Test Split (60/20/20) ====================
X_train_cic, X_temp, y_train_cic, y_temp = train_test_split(
    X_cic_raw, y_cic_raw, test_size=0.40, stratify=y_cic_raw, random_state=42
)
X_val_cic, X_test_cic, y_val_cic, y_test_cic = train_test_split(
    X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=42
)

print(f"Train: {X_train_cic.shape} | Val: {X_val_cic.shape} | Test: {X_test_cic.shape}")

Train: (1511193, 73) | Val: (503731, 73) | Test: (503731, 73)


In [13]:
# ==================== B.3 SMOTE on Train Only (minority classes only) ====================
counts = np.bincount(y_train_cic)
strategy = {i: max(cnt, 5000) for i, cnt in enumerate(counts) if 0 < cnt < 5000}
if strategy:
    print(f"Applying SMOTE to {len(strategy)} minority classes")
    sm = SMOTE(sampling_strategy=strategy, random_state=42, k_neighbors=3)
    X_train_res_cic, y_train_res_cic = sm.fit_resample(X_train_cic, y_train_cic)
else:
    X_train_res_cic, y_train_res_cic = X_train_cic, y_train_cic

print(f"Train after SMOTE: {X_train_res_cic.shape}")
print(f"Class distribution: {np.bincount(y_train_res_cic)}")

Applying SMOTE to 7 minority classes
Train after SMOTE: (1533137, 73)
Class distribution: [1257034    5000   76808    6172  103707    5000    5000    5000    5000
    5000   54416    5000]


In [14]:
# ==================== B.4 Reshape for Conv1D: (n, features, 1) ====================
X_train_cic_3d = X_train_res_cic.reshape(len(X_train_res_cic), -1, 1)
X_val_cic_3d   = X_val_cic.reshape(len(X_val_cic), -1, 1)
X_test_cic_3d  = X_test_cic.reshape(len(X_test_cic), -1, 1)

print(f"3D shapes — Train: {X_train_cic_3d.shape} | Val: {X_val_cic_3d.shape} | Test: {X_test_cic_3d.shape}")

3D shapes — Train: (1533137, 73, 1) | Val: (503731, 73, 1) | Test: (503731, 73, 1)


In [15]:
# ==================== B.5 Save CICIDS2017 Model‑Ready Artifacts ====================
np.save(os.path.join(OUT_DIR, "cic_Xtrain_3d.npy"), X_train_cic_3d)
np.save(os.path.join(OUT_DIR, "cic_ytrain.npy"),    y_train_res_cic)
np.save(os.path.join(OUT_DIR, "cic_Xval_3d.npy"),   X_val_cic_3d)
np.save(os.path.join(OUT_DIR, "cic_yval.npy"),       y_val_cic)
np.save(os.path.join(OUT_DIR, "cic_Xtest_3d.npy"),  X_test_cic_3d)
np.save(os.path.join(OUT_DIR, "cic_ytest.npy"),      y_test_cic)

# Flat versions for baselines (Random Forest)
np.save(os.path.join(OUT_DIR, "cic_Xtrain_flat.npy"), X_train_res_cic)
np.save(os.path.join(OUT_DIR, "cic_Xval_flat.npy"),   X_val_cic)
np.save(os.path.join(OUT_DIR, "cic_Xtest_flat.npy"),  X_test_cic)

joblib.dump(feature_names_cic, os.path.join(OUT_DIR, "cic_feature_names.pkl"))
joblib.dump(class_names_cic,   os.path.join(OUT_DIR, "cic_classes.pkl"))
print("CICIDS2017 model‑ready artifacts saved ✓")

CICIDS2017 model‑ready artifacts saved ✓


# # ✅ Preprocessing Complete!
# 
# All model‑ready arrays are in `/kaggle/working/processed/`.
# 
# **Next → Baseline Models (Random Forest, CNN, GRU)** then **LCGA training**.